# Threshold And Calibration Blocks

Этот ноутбук содержит отдельные ячейки, которые можно вставить в MVP train notebook после блока `OOF Evaluation` и до `OOT Validation`.

Ожидается, что в основном ноутбуке уже существуют переменные: `df`, `target`, `score`, `final_threshold`, `RANDOM_STATE`, `func`, `StratifiedKFold`, `precision_score`, `recall_score`, `f1_score`, `fbeta_score`, `sk_roc_auc_score`, `sk_average_precision_score`, `sk_brier_score_loss`.

## Imports

Добавь в import-ячейку основного ноутбука, если этих импортов там еще нет.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline

## Threshold Search

Подбор threshold по `F-beta`, как в оригинальном исследовательском ноутбуке.

In [ ]:
def find_threshold(y_true, y_pred, eval_function, **kwargs):
    score_by_threshold = {}

    for threshold_value in np.arange(0.05, 1, 0.05):
        y_class = y_pred > threshold_value
        score_by_threshold[threshold_value] = eval_function(y_true, y_class, **kwargs)

    return max(score_by_threshold.items(), key=lambda x: x[1])[0]


t = df[target]
s = df[score]

threshold_rows = []

for beta in [0.35, 0.5, 1, 2]:
    threshold_value = find_threshold(t, s, fbeta_score, beta=beta)
    y_class = s > threshold_value

    threshold_rows.append({
        'beta': beta,
        'threshold': threshold_value,
        'selected_share': y_class.mean(),
        'precision': precision_score(t, y_class),
        'recall': recall_score(t, y_class),
        'f1': f1_score(t, y_class),
        'fbeta': fbeta_score(t, y_class, beta=beta),
    })

threshold_report = pd.DataFrame(threshold_rows)
display(threshold_report)

## Optional Threshold Update

Если после анализа нужно заменить фиксированный threshold на оптимальный по F1, выполни эту ячейку.

In [ ]:
final_threshold = threshold_report.loc[
    threshold_report['beta'].eq(1),
    'threshold',
].iloc[0]

print('Selected final_threshold:', final_threshold)

## Calibration: Platt / Spline / Isotonic

Калибраторы обучаются по CV на OOF-скорах `df[score]`, как в оригинальной модели.

In [ ]:
platt = LogisticRegression(C=1e10)
platt_spline = make_pipeline(
    SplineTransformer(n_knots=5, degree=3),
    LogisticRegression(C=1.0),
)
isotonic = IsotonicRegression(out_of_bounds='clip')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X = pd.DataFrame(df[score])
y = df[target]
z = df['strat']

X_scored = pd.DataFrame()

for i, (train_idx, val_idx) in enumerate(skf.split(X, z), start=1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val = X.iloc[val_idx].copy()

    print(f'Calibrating on fold {i}...')

    platt.fit(X_train, y_train)
    platt_spline.fit(X_train, y_train)
    isotonic.fit(X_train, y_train)

    X_val[f'{score}_platt'] = platt.predict_proba(X_val)[:, 1]
    X_val[f'{score}_platt_spline'] = platt_spline.predict_proba(X_val)[:, 1]
    X_val[f'{score}_isotonic'] = isotonic.transform(X_val)

    X_scored = pd.concat([X_scored, X_val], axis=0)

calibrated_cols = [f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']
df[calibrated_cols] = X_scored.sort_index()[calibrated_cols]

display(df[[score] + calibrated_cols].head())

## Calibration Metrics

Сравнение raw score и калиброванных вариантов.

In [ ]:
calibration_metrics = []

for score_col in [
    score,
    f'{score}_platt',
    f'{score}_platt_spline',
    f'{score}_isotonic',
]:
    calibration_metrics.append({
        'score': score_col,
        'roc_auc': sk_roc_auc_score(df[target], df[score_col]),
        'pr_auc': sk_average_precision_score(df[target], df[score_col]),
        'ap_gain': sk_average_precision_score(df[target], df[score_col]) - df[target].mean(),
        'brier': sk_brier_score_loss(df[target], df[score_col]),
    })

calibration_metrics = pd.DataFrame(calibration_metrics).sort_values('brier')
display(calibration_metrics)

## Reliability Diagram With Brier Decomposition

Повторяет блок из оригинального `train_rnd.ipynb`: reliability diagram + decomposition на Reliability / Resolution / Uncertainty.

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', alpha=0.5)

models = [
    (df[score], 'Raw', 'blue'),
    (df[f'{score}_platt'], 'Platt', 'green'),
    (df[f'{score}_platt_spline'], 'Platt Spline', 'orange'),
    (df[f'{score}_isotonic'], 'Isotonic', 'red'),
]

for pred, name, color in models:
    rel, res, unc = func.brier_decomposition(df[target], pred)
    bs = rel - res + unc

    f_pos, m_val = calibration_curve(
        df[target],
        pred,
        n_bins=10,
        strategy='quantile',
    )

    label = (
        f'{name}\n'
        f'  BS: {bs:.4f}\n'
        f'  Rel: {rel:.4f}\n'
        f'  Res: {res:.4f}\n'
        f'  Unc: {unc:.4f}'
    )

    plt.plot(m_val, f_pos, 's-', color=color, label=label)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Reliability Diagram with Brier Decomposition')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Score Distributions For Calibrated Scores

In [ ]:
for score_col in [
    score,
    f'{score}_platt',
    f'{score}_platt_spline',
    f'{score}_isotonic',
]:
    plt.figure(figsize=(8, 5))
    sns.histplot(
        df[df[target] == 0][score_col],
        label='target=0',
        kde=True,
        stat='density',
        color='blue',
        alpha=0.4,
    )
    sns.histplot(
        df[df[target] == 1][score_col],
        label='target=1',
        kde=True,
        stat='density',
        color='orange',
        alpha=0.4,
    )
    plt.title(f'Score distribution: {score_col}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## Optional: Deciles For Calibrated Scores

Если нужно сравнить, меняется ли ранжирование после калибровки.

In [ ]:
for score_col in [
    score,
    f'{score}_platt',
    f'{score}_platt_spline',
    f'{score}_isotonic',
]:
    print(score_col)
    display(decile_report(df, score_col, target))